# Hansen Ch.12 Instrumental Variables — 计算

完整理论见 `Hansen_Ch12_Exercises_Solutions.md`（**12.1–12.28**）。

本 notebook：**12.23 AJR**、**12.25 Card**、**12.27 AK 黑人子样本（3 QOB）**。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def ols(y, X):
    b = inv(X.T @ X) @ (X.T @ y)
    return b, y - X @ b

def tsls(y, X, Z):
    PZ = Z @ inv(Z.T @ Z) @ Z.T
    b = inv(X.T @ PZ @ X) @ (X.T @ PZ @ y)
    e = y - X @ b
    Xh = PZ @ X
    meat = (Xh * e[:, None]).T @ (Xh * e[:, None])
    V = inv(X.T @ PZ @ X) @ meat @ inv(X.T @ PZ @ X)
    return b, e, V

def se_hom(X, e):
    n, k = X.shape
    return np.sqrt(np.diag((e @ e / (n - k)) * inv(X.T @ X)))


## Exercise 12.23 AJR2001

In [ ]:

ajr = pd.read_excel(ROOT / "AJR2001/AJR2001.xlsx")
d = ajr[["loggdp", "risk", "logmort0"]].dropna()
y, risk, lm = d.loggdp.values, d.risk.values, d.logmort0.values
n = len(d)
print("n =", n)

# OLS
X = np.column_stack([risk, np.ones(n)])
b, e = ols(y, X)
print("OLS risk:", b[0], "hom SE", se_hom(X, e)[0], "HC", np.sqrt(tsls(y, X, X)[2][0,0]))

# RF
Z = np.column_stack([lm, np.ones(n)])
g, ug = ols(risk, Z)
print("RF logmort:", g[0], "hom SE", se_hom(Z, ug)[0])

# 2SLS / ILS / CF
biv, eiv, Viv = tsls(y, X, Z)
print("2SLS:", biv, "robust SE", np.sqrt(np.diag(Viv)))
print("ILS:", (ols(y, Z)[0][0] / g[0]))
u = risk - Z @ g
bcf, _ = ols(y, np.column_stack([risk, u, np.ones(n)]))
print("Control function (beta_risk, gamma_u, const):", bcf)

# + latitude, africa
d2 = ajr[["loggdp", "risk", "logmort0", "latitude", "africa"]].dropna()
X2 = np.column_stack([d2.risk, d2.latitude, d2.africa, np.ones(len(d2))])
Z2 = np.column_stack([d2.logmort0, d2.latitude, d2.africa, np.ones(len(d2))])
print("OLS +lat+africa:", ols(d2.loggdp.values, X2)[0])
print("2SLS +lat+africa:", tsls(d2.loggdp.values, X2, Z2)[0])

# logmort + square instruments
Z3 = np.column_stack([lm, lm**2, np.ones(n)])
b3, e3, V3 = tsls(y, X, Z3)
print("2SLS (logmort, logmort^2):", b3, "SE", np.sqrt(np.diag(V3)))
# FS F
e_fs = risk - Z3 @ inv(Z3.T @ Z3) @ (Z3.T @ risk)
e_r = risk - risk.mean()
F = ((e_r@e_r - e_fs@e_fs)/2) / (e_fs@e_fs/(n-3))
print("FS F:", F)
J = (e3 @ Z3 @ inv(Z3.T@Z3) @ Z3.T @ e3) / (e3@e3/n)
print("Sargan J:", J, "p=", 1-stats.chi2.cdf(J, 1))


## Exercise 12.25 Card1995 (2SLS with nearc4a, nearc4b)

In [ ]:

card = pd.read_excel(ROOT / "Card1995/Card1995.xlsx")
card["exper"] = card["age76"] - card["ed76"] - 6
card["exp2"] = (card["exper"] ** 2) / 100
cols = ["lwage76","ed76","exper","exp2","black","smsa76r","reg76r","nearc4a","nearc4b","nearc2"]
d = card[cols].apply(pd.to_numeric, errors="coerce").dropna()
y = d.lwage76.values
Xexo = np.column_stack([d.exper, d.exp2, d.black, d.smsa76r, d.reg76r, np.ones(len(d))])
X = np.column_stack([d.ed76.values, Xexo])
Z = np.column_stack([d.nearc4a, d.nearc4b, Xexo])
b, e, V = tsls(y, X, Z)
print("n=", len(d), "edu 2SLS=", b[0], "SE=", np.sqrt(V[0,0]))
edu = d.ed76.values
e_fs = edu - Z @ inv(Z.T@Z) @ (Z.T @ edu)
e_r = edu - Xexo @ inv(Xexo.T@Xexo) @ (Xexo.T @ edu)
n, k, q = len(d), Xexo.shape[1], 2
F = ((e_r@e_r - e_fs@e_fs)/q) / (e_fs@e_fs/(n-k-q))
print("FS F (2 excl. instruments):", F)


## Exercise 12.27 AK1991 Black men, 3 QOB instruments

In [ ]:

ak = pd.read_excel(ROOT / "AK1991/AK1991.xlsx")
blk = ak[ak.black == 1].copy()
yob_d = pd.get_dummies(blk.yob, prefix="yob", drop_first=True)
reg_d = pd.get_dummies(blk.region, prefix="reg", drop_first=True)
Zex = pd.get_dummies(blk.qob, prefix="qob", drop_first=True)
Xexo = np.column_stack([blk.smsa.values, blk.married.values, yob_d.values, reg_d.values, np.ones(len(blk))])
edu = blk.edu.values.astype(float)
y = blk.logwage.values.astype(float)
X = np.column_stack([edu, Xexo])
Z = np.column_stack([Zex.values.astype(float), Xexo])
mask = np.isfinite(X).all(1) & np.isfinite(y) & np.isfinite(Z).all(1)
X, y, Z, edu, Xexo = X[mask], y[mask], Z[mask], edu[mask], Xexo[mask]
b, e, V = tsls(y, X, Z)
print("Black n=", len(y), "edu 2SLS=", b[0], "SE=", np.sqrt(V[0,0]))
e_fs = edu - Z @ inv(Z.T@Z) @ (Z.T @ edu)
e_r = edu - Xexo @ inv(Xexo.T@Xexo) @ (Xexo.T @ edu)
q = Zex.shape[1]
n, k = len(y), Xexo.shape[1]
F = ((e_r@e_r - e_fs@e_fs)/q) / (e_fs@e_fs/(n-k-q))
print("FS F (3 QOB):", F, " (weak if << 10)")
